In [19]:
!pip install -q qdrant-client open_clip_torch

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [20]:
import os
import glob
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import open_clip

# CONFIG
IMAGE_DIR = "/kaggle/input/datasets/js042710/image-visual-search/data"      # thư mục chứa ảnh
IMAGE_EXTS = (".jpg")
QDRANT_PATH = "/kaggle/working/qdrant_db"          # Qdrant lưu dữ liệu (embedded, không cần server)
COLLECTION_NAME = "image_search"
MODEL_NAME = "ViT-B-32"
PRETRAINED = "openai"
BATCH_SIZE = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [ ]:
# 1. Load CLIP model
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE).eval()

EMBED_DIM = model.visual.output_dim
print("Embedding dim:", EMBED_DIM)

In [ ]:
# 2. Gom danh sách ảnh trong dataset
image_paths = []
for ext in IMAGE_EXTS:
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, "**", f"*{ext}"), recursive=True))
image_paths = sorted(image_paths)
print(f"Tổng số ảnh tìm thấy: {len(image_paths)}")
assert len(image_paths) > 0, "Khong tim thay anh nao, kiem tra lai IMAGE_DIR"


In [ ]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert("RGB")
            img = self.transform(img)
        except Exception as e:
            print(f"Lỗi đọc ảnh {path}: {e}")
            img = torch.zeros(3, 224, 224)
        return img, idx

dataset = ImageDataset(image_paths, preprocess)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# 3. Tạo collection trên Qdrant (local/embedded)
client = QdrantClient(path=QDRANT_PATH)

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
)
print("Đã tạo collection:", COLLECTION_NAME)

In [ ]:
# 4. Encode toàn bộ ảnh và upsert vào Qdrant
with torch.no_grad():
    for imgs, idxs in tqdm(loader, desc="Encoding + upserting"):
        imgs = imgs.to(DEVICE)
        feats = model.encode_image(imgs)
        feats = F.normalize(feats, dim=-1)  # L2 normalize -> cosine similarity = dot product
        feats = feats.cpu().numpy()

        points = [
            PointStruct(
                id=int(idxs[i]),
                vector=feats[i].tolist(),
                payload={"path": image_paths[int(idxs[i])]},
            )
            for i in range(len(idxs))
        ]
        client.upsert(collection_name=COLLECTION_NAME, points=points)

print("Đã encode & upsert xong", len(image_paths), "ảnh")

In [ ]:
# 5. Search: đưa ảnh mới vào để tìm ảnh tương đồng
def search_by_image(query_image_path, top_k=5):
    img = Image.open(query_image_path).convert("RGB")
    img_t = preprocess(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        feat = model.encode_image(img_t)
        feat = F.normalize(feat, dim=-1)
    query_vector = feat.cpu().numpy()[0].tolist()

    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,   
        limit=top_k,
    )
    return response.points    # trả về list ScoredPoint, giữ nguyên cấu trúc payload/score như cũ

def show_results(query_image_path, results):
    n = len(results) + 1
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))

    query_img = Image.open(query_image_path).convert("RGB")
    axes[0].imshow(query_img)
    axes[0].set_title("Query")
    axes[0].axis("off")

    for i, res in enumerate(results):
        path = res.payload["path"]
        score = res.score
        img = Image.open(path).convert("RGB")
        axes[i + 1].imshow(img)
        axes[i + 1].set_title(f"score={score:.3f}")
        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.show()


QUERY_IMAGE_PATH = "/kaggle/input/datasets/js042710/test-image/1783304738068_275658667048781604_8456278439376730285_0c06bb49bd0ba4375feed9ebe8c6291b.jpg"  # <-- đổi thành đường dẫn ảnh bạn muốn tìm

results = search_by_image(QUERY_IMAGE_PATH, top_k=5)
show_results(QUERY_IMAGE_PATH, results)